**26 | Final Development Model Training, Reproducible Inference and Qualified Test Gate**

**Human Movement Intelligence | Reza Saadatyar (2026)**

This notebook implements the Stage 25 **candidate protocol** without automatically opening the reserved Test set. Its default, safe path fits one final 1D-ResNet and one Random Forest reference on all 29 Development participants, verifies Train-only preprocessing, saves both models and reloads them for reproducible inference. **No original S023/S032 source file or Test label is loaded unless the separate, explicit protocol approval gate is satisfied.** The previous descriptive inspection of these participants must always be disclosed if a qualified held-out report is later authorised.

In [1]:
%cd /content

!git clone https://github.com/RezaSaadatyar/human-movement-intelligence.git

%cd /content/human-movement-intelligence

from pathlib import Path
root = Path.cwd().resolve()
required_file = root / "code/python/src/core/data_loading.py"
print("Project root:", root)
print("Required file exists:", required_file.is_file())

/content
Cloning into 'human-movement-intelligence'...


remote: Enumerating objects: 1357, done.
remote: Counting objects: 100% (269/269), done.
remote: Compressing objects: 100% (227/227), done.
remote: Total 1357 (delta 168), reused 43 (delta 27), pack-reused 1088 (from 2)
Receiving objects: 100% (1357/1357), 3.70 MiB | 9.40 MiB/s, done.
Resolving deltas: 100% (818/818), done.
/content/human-movement-intelligence
Project root: /content/human-movement-intelligence
Required file exists: True


**01. Scope, Approval Boundary and Reproducibility Setup**

The candidate procedure fixes HARTH v2.0 source revision, 100-sample time-safe pure-label windows at 50 Hz, 50-sample stride, eight original source activities, 500-window chronological cap per subject, 14,201 Development windows and 29 Development people. Final 1D-ResNet uses seed 42, 400 Adam updates, batch size 128, inverse-square-root Weighted Cross-Entropy and Development-fitted channel scaling. The fixed RF reference has 120 trees, minimum leaf size 3 and balanced-subsample weights on 30 features. Stage 25's manifest currently says `PENDING_USER_REVIEW`, so Test access is disabled.

In [ ]:
# ==================== 01. Environment and Approval-Safe Protocol ====================

# 1. Locate the repository and load the already reviewed scientific settings.
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
root=Path.cwd().resolve()
while not (root/"code/python/src/core/harth_final_inference.py").is_file():
    if root.parent==root:
        raise RuntimeError("Run within the HMI repository.")
    root=root.parent
sys.path.insert(0,str(root/"code/python/src"))
sys.path.insert(0,str(root/"studies/wearable_activity_recognition/src"))
from core.harth_final_inference import (
    load_final_protocol, fit_final_resnet, fit_development_scaler,
    apply_frozen_scaler, save_final_models, load_final_models,
    infer_source_windows, enforce_qualified_test_approval,
    score_qualified_heldout,
)
from harth_data import (
    HARTH_SUBJECTS, SENSOR_COLUMNS, HARTH_REVISION, ACTIVITY_NAMES,
    prepare_harth_window_cohort,
)
from core.feature_engineering import extract_signal_features
from sklearn.ensemble import RandomForestClassifier
import torch

# 2. Validate original code hashes; Test is NOT authorised by running this cell.
PROPOSAL=root/"studies/wearable_activity_recognition/experiments/stage_25_protocol_proposal.json"
protocol=load_final_protocol(PROPOSAL,root,require_approval=False)
RESERVED_TEST={"S023","S032"}
DEVELOPMENT=sorted(set(HARTH_SUBJECTS)-RESERVED_TEST)
CLASSES=tuple(protocol["source"]["task_codes"])
ARTIFACT_DIR=root/"outputs/harth_stage26_final"
assert len(DEVELOPMENT)==29 and HARTH_REVISION==protocol["source"]["revision"]
assert len(SENSOR_COLUMNS)==6

# 3. Prefer Colab CUDA for training; record the true accelerator environment.
if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU in Colab: Runtime > Change runtime type > GPU.")
DEVICE="cuda"
torch.set_float32_matmul_precision("highest")
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False
print("\n"+"="*70)
print("STAGE 26 | SAFE FINAL DEVELOPMENT MODEL FIT")
print("="*70)
print("GPU:",torch.cuda.get_device_name(0),"| PyTorch:",torch.__version__)
print("Source revision:",HARTH_REVISION)
print("Protocol approval status:",protocol["status"])
print("Reserved Test loaded: NO")
print("="*70)

**02. Original 29-Person Development Cohort and Source Integrity**

Construct the exact source-row-aligned 500-cap cohort from the official pinned recordings, applying strict time continuity and label-purity checks **before** filtering out-of-task annotations. Do not use `S023` or `S032` for fitting, feature extraction, batch-normalisation statistics, class weights or architecture decisions. The manifest's Development SHA256 is compared with the exact reconstructed sample used in Stages 23–25.

In [ ]:
# ==================== 02. Development-Only Source Cohort ====================

# 1. Recreate genuine original Development source windows.
X_dev,y_dev,dev_metadata,dev_overview=prepare_harth_window_cohort(
    subjects=DEVELOPMENT,data_dir=root/"data/raw/harth_v2",
    task_classes=CLASSES,max_windows_per_subject=500,
    window_size=100,step_size=50,sampling_rate=50,
)
assert X_dev.shape==(14201,6,100)
assert len(y_dev)==len(dev_metadata)
assert set(dev_metadata["subject_id"])==set(DEVELOPMENT)
assert not(set(dev_metadata["subject_id"])&RESERVED_TEST)

# 2. Recalculate the same protocol fingerprint without importing Test data.
from core.temporal_benchmark import make_benchmark_plan
dev_plan=make_benchmark_plan(
    X_dev,y_dev,dev_metadata["subject_id"].to_numpy(),
    classes=CLASSES,n_splits=3,forbidden_subjects=RESERVED_TEST,
)
assert dev_plan["fingerprint"]==protocol["source_development_sha256"]
display(dev_overview[["subject_id","eligible_windows","sampled_windows"]])
print("Development SHA256 verified:",dev_plan["fingerprint"])
print("Actual Test source files opened: NO")

**03. Development-Only Channel Scaling and Source Class Weights**

The six original sensor axes must remain in the same order in every training and inference window. Channel statistics are calculated from *all final Development windows only*, not from the protected Test. Weighted Cross-Entropy receives Train-only class weights based on actual original activity codes. These parameters are not tuned using held-out observations.

In [ ]:
# ==================== 03. Freeze Development Fitted Preprocessing ====================

# 1. Check the exact sensor order and Development-only fitted scaler.
scaler=fit_development_scaler(X_dev)
X_scaled=apply_frozen_scaler(X_dev,scaler)
assert X_scaled.shape==X_dev.shape and np.isfinite(X_scaled).all()
stats=pd.DataFrame({
    "channel":SENSOR_COLUMNS,
    "development_mean":scaler["mean"],
    "development_std":scaler["std"],
})
display(stats.round(6))

# 2. Describe class support; these are overlapping source-window counts.
source_counts=pd.Series(y_dev).value_counts().reindex(CLASSES,fill_value=0)
display(pd.DataFrame({
    "source_code":CLASSES,
    "activity":[ACTIVITY_NAMES[c] for c in CLASSES],
    "development_windows":source_counts.to_numpy(),
}))
assert (source_counts>0).all()

**04. Train the Final 1D-ResNet Using the Frozen 400-Update Procedure**

A newly initialised 1D-ResNet trains on the entire eligible Development cohort using seed 42, batch size 128, Adam 0.001 and inverse-square-root class weighting. It is deliberately trained for exactly 400 optimiser updates and the last-step weights are retained. There is no Validation or Test-driven checkpoint selection. The training loss trace is a diagnostic, **not** an unbiased accuracy estimate.

In [ ]:
# ==================== 04. Final Development-Only Neural Fitting ====================

# 1. Fit exactly one frozen neural model with no original Test labels.
model,trained_scaler,train_history,class_weights,train_time=fit_final_resnet(
    X_dev,y_dev,protocol,device=DEVICE
)

# 2. Verify the final training budget and visualise real Train-loss observations.
assert int(train_history["step"].iloc[-1])==400
display(train_history)
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(train_history["step"],train_history["development_train_loss"],marker="o")
ax.set(title="Final 1D-ResNet | Development Train loss only",
       xlabel="Optimizer update",ylabel="Weighted Cross-Entropy loss")
ax.grid(alpha=.2)
fig.tight_layout()
plt.show()
print("Final model fit duration (seconds):",round(train_time,2))
print("Test data used in training: NO")

**05. Fit the Frozen Random Forest as a Classical Reference**

The Random Forest reference uses 30 deterministic features from the same eligible Development source windows. Feature extraction independently summarises each six-channel window and does not fit a Test-derived transform. The 120-tree specification is frozen from Stage 25, not selected after seeing Test performance.

In [ ]:
# ==================== 05. Train the Fixed Classical Reference ====================

# 1. Extract 30 source-window features with documented and ordered names.
features=extract_signal_features(X_dev,channel_names=SENSOR_COLUMNS)
assert features.shape==(14201,30)

# 2. Fit the exact 120-tree RF specification on Development source labels.
rf=RandomForestClassifier(
    n_estimators=120,min_samples_leaf=3,
    class_weight="balanced_subsample",random_state=42,n_jobs=-1
)
rf.fit(features.to_numpy(),y_dev)
assert set(rf.classes_)==set(CLASSES)
print("RF trees:",len(rf.estimators_))
print("RF feature count:",rf.n_features_in_)
print("RF Train subjects:",len(DEVELOPMENT))
print("RF Test subjects accessed: NO")

**06. Save Both Models, Class Mapping and Original Sensor Statistics**

A reproducible inference artefact must include more than PyTorch weights: the channel order, exact class-code map, windowing assumptions, six Development-fitted means and SDs, RF feature-column order, source fingerprint and protocol digests are saved as accompanying JSON. The original source recordings and person-level Test labels are not embedded in the artefacts. The saved RF `joblib` file must only be loaded from a trusted, locally created location.

In [ ]:
# ==================== 06. Versioned Local Model Artefacts ====================

# 1. Save trusted weights, scaler, source labels and reference forest metadata.
artifact_info=save_final_models(
    model,trained_scaler,rf,list(features.columns),
    protocol,dev_plan["fingerprint"],ARTIFACT_DIR
)
display(pd.DataFrame([
    {"artifact":name,"path":path,"sha256":artifact_info["sha256"][name]}
    for name,path in artifact_info.items() if name!="sha256"
]))
print("Model artefacts stored locally:",ARTIFACT_DIR)
print("Artefacts are ignored by Git and should not be committed.")

**07. Reload Exact Artefacts and Verify Prediction Consistency**

Test the complete reproducible inference pathway on Development source windows *only*. This round-trip check catches missing weights, incompatible input shapes, incorrect class labels, channel-order errors and forgotten normalisation statistics. Agreement with in-memory predictions is an engineering requirement, not evidence that the model will generalise to new participants.

In [ ]:
# ==================== 07. Development Inference Round-Trip Check ====================

# 1. Load saved models and preprocessing from the newly written trusted directory.
loaded_model,loaded_rf,config=load_final_models(ARTIFACT_DIR,device="cpu")
N_CHECK=24
unseen_style_windows=X_dev[:N_CHECK]
reloaded_nn,reloaded_rf=infer_source_windows(
    unseen_style_windows,loaded_model,loaded_rf,config,device="cpu"
)

# 2. Verify original class codes and compare RF reload to the original fitted model.
assert len(reloaded_nn)==len(reloaded_rf)==N_CHECK
assert set(reloaded_nn).issubset(CLASSES)
assert set(reloaded_rf).issubset(CLASSES)
direct_rf=rf.predict(
    features.iloc[:N_CHECK].to_numpy()
)
np.testing.assert_array_equal(reloaded_rf,direct_rf)
display(pd.DataFrame({
    "true_dev_code":y_dev[:N_CHECK],
    "reloaded_resnet_prediction":reloaded_nn,
    "reloaded_rf_prediction":reloaded_rf,
}))
print("Saved neural and classical inference: PASSED")
print("Independent Test model performance assessed: NO")

**08. Security and Test-Gate Review (Non-Executing by Default)**

**No Test source access occurs in this notebook by default.** The existing Stage 25 JSON still has `PENDING_USER_REVIEW`. To authorise Test evaluation in a later review, the user must explicitly confirm the source cohort, 500-window sampling policy for both original reserved participants, eight-class metrics, original descriptive-inspection limitation, and the decision to report one frozen primary model plus RF reference. A passing Development training Quality Gate is not equivalent to Test authorisation.

In [ ]:
# ==================== 08. Demonstrate Test Gate WITHOUT Source Access ====================

# 1. Check the current approval state and explain why Test is not accessible.
print("Protocol status:",protocol["status"])
print("Qualified Test requested in this notebook: NO")
if protocol["status"]!="APPROVED":
    print("Qualified Test gate: BLOCKED pending explicit user review.")
else:
    print("Qualified Test gate: approval recorded; separate confirmation still required.")

# 2. Do not load, download, enumerate or score either reserved recording.
assert RESERVED_TEST=={"S023","S032"}
print("S023 and S032 recordings read in this notebook: NO")

**09. Define the Qualified Held-Out Report, Without Running It**

After approval, Stage 26 may be extended to a separately gated Test execution using the same source-window preprocessing, exactly 500 eligible post-windowing samples per reserved person, and existing frozen models. Both Test subjects must be reported separately, along with fixed-eight-class Macro F1, Balanced Accuracy, overall Accuracy, class precision/recall/F1/support and confusion matrices. No Test-driven changes or selecting whichever model scores higher are permitted. Previously inspecting the reserved recordings and their same-study provenance preclude calling this an independent blind external benchmark.

In [ ]:
# ==================== 09. Explicit Qualified Test Reporting Contract ====================

# 1. Show planned metrics only, not hypothetical Test scores.
report_plan=pd.DataFrame([
    {"evaluation":"Primary metric","requirement":"Fixed eight-class Macro F1"},
    {"evaluation":"Secondary metrics","requirement":"Balanced accuracy and overall accuracy"},
    {"evaluation":"Per-class diagnostics","requirement":"Precision, recall, F1, actual support"},
    {"evaluation":"Independent people","requirement":"S023 and S032 reported separately"},
    {"evaluation":"Model comparison","requirement":"Frozen ResNet and predeclared RF reference"},
    {"evaluation":"Forbidden operations","requirement":"No Test-driven refitting, model choice or hyperparameter search"},
    {"evaluation":"Scientific limitation","requirement":"Originally descriptively inspected; same-study, not external"},
])
display(report_plan)
print("Test metrics created by this stage: NONE")

**10. Reproducibility Summary and Final Development Quality Gate**

A passing gate here verifies that both **final Development-fitted** models exist, their frozen metadata are present and reloadable, source-window/label correspondence remains intact and no protected Test participant was loaded. This does not verify a reserved-Test generalisation claim. Approval remains a separate human decision; Stage 27's report must distinguish trained artefact readiness from externally validated performance.

In [ ]:
# ==================== 10. Final Training and Inference Quality Gate ====================

# 1. Confirm exact source cohort, frozen controls and trained model artefacts.
assert X_dev.shape==(14201,6,100)
assert set(dev_metadata["subject_id"])==set(DEVELOPMENT)
assert not(set(dev_metadata["subject_id"])&RESERVED_TEST)
assert protocol["primary_model"]["optimizer_steps"]==400
assert train_history["step"].iloc[-1]==400
assert len(class_weights)==8 and np.isfinite(class_weights).all()
assert all(Path(path).is_file() for name,path in artifact_info.items() if name!="sha256")

# 2. Verify complete raw-window inference with Development-fitted statistics.
assert config["source_codes"]==list(CLASSES)
assert config["sensor_columns"]==SENSOR_COLUMNS
assert config["preprocessing_fit_scope"]=="Development only"
assert not config["test_fitted_values"]
np.testing.assert_array_equal(reloaded_rf,direct_rf)

# 3. Report clearly which objectives have, and have NOT, been achieved.
print(""+"="*70)
print("STAGE 26 | FROZEN DEVELOPMENT FIT AND INFERENCE ARTEFACTS")
print("="*70)
print("Frozen 29-person Development cohort: VERIFIED")
print("Primary 1D-ResNet trained: YES, 400 fixed updates")
print("Fixed Random Forest trained: YES, 120 trees")
print("Train-only scaler, class map and channel order: SAVED")
print("Trusted artifact reloading/inference: PASSED")
print("Original S023/S032 Test accessed: NO")
print("-"*70)
print("Stage 26 Development training/inference quality gate: PASSED")
print("Protocol approval status:",protocol["status"])
print("Qualified Test evaluation performed: NO")
print("Next: explicit qualified-Test approval and separate gated scoring")
print("="*70)